In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                 "faiss-cpu", "sentence-transformers", "pandas", "numpy", "pyarrow", 
                 "transformers", "accelerate", "bitsandbytes"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-U', 'faiss-cpu', 'sentence-transformers', 'pandas', 'numpy', 'pyarrow', 'transformers', 'accelerate', 'bitsandbytes'], returncode=0)

In [13]:
def try_download(path):
    """Trigger a browser download if running in Colab; otherwise just confirm the file was saved
    (Kaggle/local/other environments can grab it from the file browser or Output pane)."""
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print(f"Saved: {path} (not on Colab — download it from the notebook's file/output browser)")

In [2]:
import time
import numpy as np
import pandas as pd
import faiss
import re
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

df = pd.read_parquet("/kaggle/input/datasets/mrnotalent/laptop-embedding/laptop_chunks_embeddings_with_lineage.parquet")
embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
device_level = df.drop_duplicates(subset="row_uid").copy()

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

index_flat = faiss.IndexFlatL2(embeddings.shape[1])
index_flat.add(embeddings)

print(f"Dataframe shape: {df.shape}")

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16 
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=bnb_config
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [3]:
def extract_filters(query: str):
    q = query.lower()
    mask = pd.Series(True, index=device_level.index)
    applied = []

    m = re.search(r"under\s*\$?(\d+)", q) or re.search(r"less than\s*\$?(\d+)", q) or re.search(r"budget.*?\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] < price
        applied.append(f"price_usd < {price}")

    m = re.search(r"over\s*\$?(\d+)", q) or re.search(r"above\s*\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] > price
        applied.append(f"price_usd > {price}")

    m = re.search(r"(\d+)\s*gb\s*ram", q) or re.search(r"at least\s*(\d+)\s*gb", q)
    if m:
        ram = float(m.group(1))
        mask &= device_level["ram_gb"] >= ram
        applied.append(f"ram_gb >= {ram}")

    if any(w in q for w in ["nvidia", "rtx", "geforce", "gaming"]):
        mask &= device_level["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)
        applied.append("gpu contains NVIDIA/RTX/GeForce")

    if "amd" in q or "ryzen" in q:
        mask &= device_level["cpu"].str.contains("Ryzen|AMD", case=False, na=False)
        applied.append("cpu contains Ryzen/AMD")

    for cpu_kw in ["i9", "i7", "i5", "i3"]:
        if cpu_kw in q:
            mask &= device_level["cpu"].str.contains(cpu_kw, case=False, na=False)
            applied.append(f"cpu contains {cpu_kw}")

    if "ssd" in q or "nvme" in q:
        mask &= device_level["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False)
        applied.append("storage is SSD/PCIe/NVMe")

    return mask, applied


def retrieve(query: str, k: int = 5):
    mask, applied_filters = extract_filters(query)
    candidate_uids = set(device_level.loc[mask, "row_uid"])

    if len(candidate_uids) < k:
        candidate_uids = set(device_level["row_uid"])
        applied_filters = applied_filters + ["(filters relaxed \u2014 too few matches)"]

    candidate_chunk_idx = df.index[df["row_uid"].isin(candidate_uids)].to_numpy()
    candidate_embeddings = embeddings[candidate_chunk_idx]

    sub_index = faiss.IndexFlatL2(candidate_embeddings.shape[1])
    sub_index.add(candidate_embeddings)

    query_vec = embed_model.encode([query]).astype("float32")
    _, local_idx = sub_index.search(query_vec, min(k, len(candidate_chunk_idx)))
    global_idx = candidate_chunk_idx[local_idx[0]]

    results = df.iloc[global_idx][
        ["title", "price_usd", "cpu", "ram_gb", "storage", "gpu", "display", "battery", "chunk_text", "lineage"]
    ]
    return results, applied_filters


def generate_recommendation(query: str, retrieved_df: pd.DataFrame):
    context_blocks = []
    for i, row in retrieved_df.iterrows():
        context_blocks.append(
            f"- {row['title']} | ${row['price_usd']:.2f} | {row['cpu']} | {row['ram_gb']}GB RAM | "
            f"{row['storage']} | {row['gpu']} | {row['display']}"
        )
    context = "\n".join(context_blocks)

    prompt = f"""You are a laptop recommendation assistant. Base your answer ONLY on the devices listed below — do not invent specs or devices not present here.

User request: {query}

Retrieved candidate devices:
{context}

Give a short recommendation: pick the best match (or top 2), and justify the choice using only the specs shown above."""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs, 
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

def recommend_uncached(query: str, k: int = 5):
    """The original, uncached pipeline — kept separate so we can benchmark against it."""
    retrieved, applied_filters = retrieve(query, k)
    answer = generate_recommendation(query, retrieved)
    return {
        "query": query,
        "filters_applied": applied_filters,
        "retrieved_context": retrieved.drop(columns=["lineage"]).to_dict(orient="records"),
        "recommendation": answer,
    }

In [4]:
class SemanticCache:
    def __init__(self, similarity_threshold: float = 0.92, max_size: int = 200):
        self.threshold = similarity_threshold
        self.max_size = max_size
        self.embeddings = []  
        self.entries = []     

    def _normalize(self, vec):
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def lookup(self, query_embedding: np.ndarray):
        if not self.embeddings:
            return None, None
        q = self._normalize(query_embedding)
        sims = np.array([np.dot(q, e) for e in self.embeddings])
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])
        if best_sim >= self.threshold:
            matched_query, response = self.entries[best_idx]
            return response, {"matched_query": matched_query, "similarity": best_sim}
        return None, None

    def store(self, query: str, query_embedding: np.ndarray, response: dict):
        if len(self.embeddings) >= self.max_size:
            self.embeddings.pop(0)
            self.entries.pop(0)
        self.embeddings.append(self._normalize(query_embedding))
        self.entries.append((query, response))

    def __len__(self):
        return len(self.entries)


cache = SemanticCache(similarity_threshold=0.92)


def recommend_cached(query: str, k: int = 5):
    query_embedding = embed_model.encode([query])[0].astype("float32")

    cached_response, match_info = cache.lookup(query_embedding)
    if cached_response is not None:
        result = dict(cached_response)
        result["cache_hit"] = True
        result["matched_query"] = match_info["matched_query"]
        result["similarity"] = round(match_info["similarity"], 4)
        return result

    result = recommend_uncached(query, k)
    result["cache_hit"] = False
    cache.store(query, query_embedding, result)
    return result

In [ ]:
benchmark_queries = [
    ("AMD Ryzen laptop under $600", 0),
    ("cheap AMD laptop under $600 dollars", 0),
    ("budget laptop with AMD Ryzen processor, under 600 bucks", 0),
    ("AMD Ryzen laptop under $600", 0),                          # exact repeat -> should hit

    ("gaming laptop with NVIDIA graphics under $1200", 1),
    ("NVIDIA gaming laptop under $1200 budget", 1),
    ("cheap gaming laptop with GeForce GPU, under $1200", 1),

    ("laptop with 32GB RAM and 1TB SSD", 2),
    ("workstation laptop, 32GB memory, 1TB SSD storage", 2),
    ("32 gigs of ram and a terabyte of SSD, what laptop fits", 2),

    ("lightweight ultrabook for travel", 3),
    ("thin and light laptop for a college student", 3),
    ("portable ultrabook good for carrying around campus", 3),

    ("cheapest laptop with an i9 processor", 4),
    ("lowest price i9 laptop", 4),
    ("budget-friendly Intel i9 laptop option", 4),

    ("reliable business laptop under $900", 5),
    ("affordable laptop for office work, under $900", 5),
    ("budget business laptop, less than 900 dollars", 5),

    ("2-in-1 convertible touchscreen laptop", 6),
    ("laptop with the best battery life for all-day use", 7),
    ("laptop good for video editing and Adobe Premiere", 8),
    ("MacBook alternative with similar build quality", 9),
    ("laptop with a numeric keypad for spreadsheet work", 10),
    ("quietest laptop fan noise under load", 11),
]
query_group_ids = [g for _, g in benchmark_queries]
benchmark_queries = [q for q, _ in benchmark_queries]  

print(f"benchmark set size: {len(benchmark_queries)} queries "
      f"({sum(1 for g in query_group_ids if query_group_ids.count(g) > 1)} in paraphrase groups, "
      f"{sum(1 for g in query_group_ids if query_group_ids.count(g) == 1)} standalone)")

print("=== Cache-enabled run ===")
cache = SemanticCache(similarity_threshold=0.92)
cached_timings = []
for q, gid in zip(benchmark_queries, query_group_ids):
    start = time.perf_counter()
    result = recommend_cached(q)
    elapsed = time.perf_counter() - start
    cached_timings.append({"query": q, "group_id": gid, "cache_hit": result["cache_hit"], "latency_s": elapsed,
                            "matched_query": result.get("matched_query"), "similarity": result.get("similarity")})
    hit_str = f"HIT (matched: \"{result.get('matched_query')}\", sim={result.get('similarity')})" if result["cache_hit"] else "MISS"
    print(f"[{elapsed:.3f}s] {hit_str:60s} | {q}")

cached_df = pd.DataFrame(cached_timings)
print()
print(f"Cache hit rate: {cached_df['cache_hit'].sum()}/{len(cached_df)} = {cached_df['cache_hit'].mean():.1%}")
print(f"Avg latency on hits:  {cached_df[cached_df['cache_hit']]['latency_s'].mean():.4f}s")
print(f"Avg latency on misses: {cached_df[~cached_df['cache_hit']]['latency_s'].mean():.4f}s")


benchmark set size: 25 queries (19 in paraphrase groups, 6 standalone)
=== Cache-enabled run ===
[36.375s] MISS                                                         | AMD Ryzen laptop under $600
[26.943s] MISS                                                         | cheap AMD laptop under $600 dollars
[36.079s] MISS                                                         | budget laptop with AMD Ryzen processor, under 600 bucks
[0.007s] HIT (matched: "AMD Ryzen laptop under $600", sim=1.0)        | AMD Ryzen laptop under $600
[39.777s] MISS                                                         | gaming laptop with NVIDIA graphics under $1200
[0.008s] HIT (matched: "gaming laptop with NVIDIA graphics under $1200", sim=0.9647) | NVIDIA gaming laptop under $1200 budget
[39.537s] MISS                                                         | cheap gaming laptop with GeForce GPU, under $1200
[30.200s] MISS                                                         | laptop with 32GB RAM 

In [6]:
print("=== Uncached run (same queries, no caching at all) ===")
uncached_timings = []
for q in benchmark_queries:
    start = time.perf_counter()
    result = recommend_uncached(q)
    elapsed = time.perf_counter() - start
    uncached_timings.append({"query": q, "latency_s": elapsed})
    print(f"[{elapsed:.3f}s] {q}")

uncached_df = pd.DataFrame(uncached_timings)
print()
print(f"Avg latency (no cache): {uncached_df['latency_s'].mean():.4f}s")
print(f"Total time (no cache):  {uncached_df['latency_s'].sum():.4f}s")
print(f"Total time (with cache): {cached_df['latency_s'].sum():.4f}s")
speedup = uncached_df["latency_s"].sum() / cached_df["latency_s"].sum()
print(f"Overall speedup from caching: {speedup:.2f}x")

=== Uncached run (same queries, no caching at all) ===
[31.301s] AMD Ryzen laptop under $600
[24.866s] cheap AMD laptop under $600 dollars
[34.027s] budget laptop with AMD Ryzen processor, under 600 bucks
[39.368s] AMD Ryzen laptop under $600
[34.288s] gaming laptop with NVIDIA graphics under $1200
[36.463s] NVIDIA gaming laptop under $1200 budget
[39.220s] cheap gaming laptop with GeForce GPU, under $1200
[14.746s] laptop with 32GB RAM and 1TB SSD
[33.322s] workstation laptop, 32GB memory, 1TB SSD storage
[13.112s] 32 gigs of ram and a terabyte of SSD, what laptop fits
[21.426s] lightweight ultrabook for travel
[37.845s] thin and light laptop for a college student
[25.841s] portable ultrabook good for carrying around campus
[26.397s] cheapest laptop with an i9 processor
[34.884s] lowest price i9 laptop
[38.942s] budget-friendly Intel i9 laptop option
[23.831s] reliable business laptop under $900
[33.521s] affordable laptop for office work, under $900
[35.748s] budget business laptop, 

In [14]:
cached_df.to_csv("semantic_cache_benchmark.csv", index=False)

summary = pd.DataFrame([{
    "hit_rate": cached_df["cache_hit"].mean(),
    "avg_latency_hit_s": cached_df[cached_df["cache_hit"]]["latency_s"].mean(),
    "avg_latency_miss_s": cached_df[~cached_df["cache_hit"]]["latency_s"].mean(),
    "total_latency_cached_s": cached_df["latency_s"].sum(),
    "total_latency_uncached_s": uncached_df["latency_s"].sum(),
    "speedup": uncached_df["latency_s"].sum() / cached_df["latency_s"].sum(),
    "similarity_threshold": 0.92,
}])
summary.to_csv("semantic_cache_summary.csv", index=False)
print(summary.to_string(index=False))

from google.colab import files
try_download("semantic_cache_benchmark.csv")
try_download("semantic_cache_summary.csv")

 hit_rate  avg_latency_hit_s  avg_latency_miss_s  total_latency_cached_s  total_latency_uncached_s  speedup  similarity_threshold
     0.08           0.007422           28.835067              663.221397                743.155922 1.120525                  0.92


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
threshold_results = []
for threshold in [0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.95, 0.98]:
    test_cache = SemanticCache(similarity_threshold=threshold)
    hits = 0
    true_positive_hits = 0
    false_positive_hits = 0
    for q, gid in zip(benchmark_queries, query_group_ids):
        query_embedding = embed_model.encode([q])[0].astype("float32")
        cached_response, match_info = test_cache.lookup(query_embedding)
        if cached_response is not None:
            hits += 1
            matched_gid = cached_response.get("_group_id") if isinstance(cached_response, dict) else None
            if matched_gid == gid:
                true_positive_hits += 1
            else:
                false_positive_hits += 1
        else:
            test_cache.store(q, query_embedding, {"query": q, "_group_id": gid, "recommendation": "placeholder"})
    threshold_results.append({
        "threshold": threshold,
        "hits": hits,
        "hit_rate": hits / len(benchmark_queries),
        "true_positive_hits": true_positive_hits,
        "false_positive_hits": false_positive_hits,
        "precision": (true_positive_hits / hits) if hits else float("nan"),
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))
threshold_df.to_csv("semantic_cache_threshold_sweep.csv", index=False)
try_download("semantic_cache_threshold_sweep.csv")


 threshold  hits  hit_rate  true_positive_hits  false_positive_hits  precision
      0.70    12      0.48                  12                    0        1.0
      0.75    12      0.48                  12                    0        1.0
      0.80    11      0.44                  11                    0        1.0
      0.85     7      0.28                   7                    0        1.0
      0.90     4      0.16                   4                    0        1.0
      0.92     2      0.08                   2                    0        1.0
      0.95     2      0.08                   2                    0        1.0
      0.98     1      0.04                   1                    0        1.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
def cosine_sim(a, b):
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    return float(np.dot(a, b))

pairs_to_check = [
    ("AMD Ryzen laptop under $600", "cheap AMD laptop under $600 dollars"),
    ("AMD Ryzen laptop under $600", "budget laptop with AMD Ryzen processor, under 600 bucks"),
    ("gaming laptop with NVIDIA graphics under $1200", "NVIDIA gaming laptop under $1200 budget"),
]
similarity_results = []
for q1, q2 in pairs_to_check:
    e1 = embed_model.encode([q1])[0]
    e2 = embed_model.encode([q2])[0]
    sim = cosine_sim(e1, e2)
    similarity_results.append({"query_a": q1, "query_b": q2, "similarity": sim})
    print(f"{sim:.4f}  |  '{q1}'  vs  '{q2}'")

pd.DataFrame(similarity_results).to_csv("semantic_cache_paraphrase_similarities.csv", index=False)

0.7976  |  'AMD Ryzen laptop under $600'  vs  'cheap AMD laptop under $600 dollars'
0.9113  |  'AMD Ryzen laptop under $600'  vs  'budget laptop with AMD Ryzen processor, under 600 bucks'
0.9647  |  'gaming laptop with NVIDIA graphics under $1200'  vs  'NVIDIA gaming laptop under $1200 budget'


In [11]:
trap_pairs = [
    ("laptop under $600", "laptop under $1600"),
    ("laptop with 16GB RAM", "laptop with 64GB RAM"),
    ("AMD Ryzen laptop under $600", "Intel Core laptop under $600"),
    ("gaming laptop with NVIDIA graphics", "gaming laptop with AMD graphics"),
    ("cheapest laptop with an i9 processor", "cheapest laptop with an i3 processor"),
    ("laptop good for video editing", "laptop good for gaming"),
]

trap_results = []
for q1, q2 in trap_pairs:
    e1 = embed_model.encode([q1])[0]
    e2 = embed_model.encode([q2])[0]
    sim = cosine_sim(e1, e2)
    trap_results.append({"query_a": q1, "query_b": q2, "similarity": sim,
                          "would_falsely_hit_at_0.92": sim >= 0.92,
                          "would_falsely_hit_at_0.80": sim >= 0.80})
    print(f"{sim:.4f}  |  '{q1}'  vs  '{q2}'")

trap_df = pd.DataFrame(trap_results)
trap_df.to_csv("semantic_cache_trap_pairs.csv", index=False)
try_download("semantic_cache_trap_pairs.csv")
print()
print(trap_df[["similarity", "would_falsely_hit_at_0.92", "would_falsely_hit_at_0.80"]].to_string())


0.8592  |  'laptop under $600'  vs  'laptop under $1600'
0.7473  |  'laptop with 16GB RAM'  vs  'laptop with 64GB RAM'
0.7386  |  'AMD Ryzen laptop under $600'  vs  'Intel Core laptop under $600'
0.7758  |  'gaming laptop with NVIDIA graphics'  vs  'gaming laptop with AMD graphics'
0.8134  |  'cheapest laptop with an i9 processor'  vs  'cheapest laptop with an i3 processor'
0.6299  |  'laptop good for video editing'  vs  'laptop good for gaming'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


   similarity  would_falsely_hit_at_0.92  would_falsely_hit_at_0.80
0    0.859184                      False                       True
1    0.747320                      False                      False
2    0.738571                      False                      False
3    0.775779                      False                      False
4    0.813412                      False                       True
5    0.629897                      False                      False


In [12]:
avg_miss = cached_df[~cached_df["cache_hit"]]["latency_s"].mean()
fair_baseline_total = avg_miss * len(cached_df)
fair_speedup = fair_baseline_total / cached_df["latency_s"].sum()

print(f"Naive cross-run speedup (in summary.csv): {summary['speedup'].iloc[0]:.2f}x  \u2014 CONFOUNDED, do not report as-is")
print(f"Fair within-run speedup estimate: {fair_speedup:.2f}x  \u2014 use this number")

corrected_summary = pd.DataFrame([{
    "hit_rate": cached_df["cache_hit"].mean(),
    "avg_latency_hit_s": cached_df[cached_df["cache_hit"]]["latency_s"].mean(),
    "avg_latency_miss_s": avg_miss,
    "fair_speedup_within_run": fair_speedup,
    "naive_speedup_cross_run_DO_NOT_USE": summary["speedup"].iloc[0],
    "similarity_threshold": 0.92,
}])
corrected_summary.to_csv("semantic_cache_summary_corrected.csv", index=False)
print(corrected_summary.to_string(index=False))

from google.colab import files
try_download("semantic_cache_paraphrase_similarities.csv")
try_download("semantic_cache_summary_corrected.csv")

Naive cross-run speedup (in summary.csv): 1.12x  — CONFOUNDED, do not report as-is
Fair within-run speedup estimate: 1.09x  — use this number
 hit_rate  avg_latency_hit_s  avg_latency_miss_s  fair_speedup_within_run  naive_speedup_cross_run_DO_NOT_USE  similarity_threshold
     0.08           0.007422           28.835067                 1.086932                            1.120525                  0.92


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>